# 🛒 ProfitPulse: Supermarket Sales & Revenue Forecasting Pipeline
### *End-to-End Retail Data Science, Anti-Leakage Feature Engineering & Multi-Model Benchmark*

**Author:** Samay Patel  
**Domain:** Retail Analytics & Predictive Business Intelligence  
**Objective:** Forecast supermarket transaction sales and profit margin metrics without target leakage, comparing 7 supervised machine learning architectures under 5-Fold Cross-Validation.

---
### 📌 Executive Summary
In retail predictive analytics, amateur models often fall into **target leakage** by incorporating accounting metrics (`COGS`, `Tax 5%`, `Gross Income`) into the feature set to predict `Total` invoice value (since mathematically, $\text{Total} = \text{COGS} + \text{Tax}$).

This notebook provides:
1. **Comprehensive EDA**: Examining customer segmentation, branch revenue, and basket size dynamics.
2. **Leak-Free Formulation**: Removing mathematical tautologies and training models exclusively on operational, product, and temporal features.
3. **Multi-Model Benchmark**: Comparing **XGBoost, Random Forest, Gradient Boosting, Ridge, Lasso, SVR, and KNN**.
4. **Production Serialization**: Packaging the winning pipeline into Scikit-Learn `ColumnTransformer` + `joblib` artifacts.


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

# Plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("✅ Libraries imported successfully.")

## 1. Data Ingestion & Overview

In [ ]:
df = pd.read_csv('supermarket_sales.csv')
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
df.info()
print("\nMissing Values Count:")
print(df.isnull().sum())

## 2. Critical Audit: Identifying & Eliminating Target Leakage
Let's mathematically verify why `Tax 5%`, `cogs`, and `gross income` cannot be used as features:
$$\text{cogs} = \text{Unit price} \times \text{Quantity}$$
$$\text{Tax 5\%} = \text{cogs} \times 0.05$$
$$\text{Total} = \text{cogs} + \text{Tax 5\%}$$
$$\text{gross income} = \text{Tax 5\%}$$


In [ ]:
# Mathematical Verification
diff_cogs = (df['cogs'] - (df['Unit price'] * df['Quantity'])).abs().max()
diff_tax = (df['Tax 5%'] - (df['cogs'] * 0.05)).abs().max()
diff_total = (df['Total'] - (df['cogs'] + df['Tax 5%'])).abs().max()

print(f"Max discrepancy in COGS formula: {diff_cogs:.2e}")
print(f"Max discrepancy in Tax formula: {diff_tax:.2e}")
print(f"Max discrepancy in Total formula: {diff_total:.2e}")
print("-> Conclusive proof that Total is an exact deterministic linear combination of COGS and Tax.")

## 3. Feature Engineering & Preprocessing Pipeline
We extract temporal dimensions (`Hour`, `DayOfWeek`, `Month`, `IsWeekend`) and isolate strictly operational attributes.


In [ ]:
# Temporal Feature Extraction
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M').dt.hour

categorical_cols = ['Branch', 'City', 'Customer type', 'Gender', 'Product line', 'Payment']
numerical_cols = ['Unit price', 'Quantity', 'Rating', 'Hour', 'DayOfWeek', 'Month', 'IsWeekend']

feature_cols = categorical_cols + numerical_cols
target_col = 'Total'

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Training observations: {len(X_train)}, Testing observations: {len(X_test)}")

## 4. Multi-Model Cross-Validation & Benchmark
We evaluate 7 candidate models with 5-Fold Cross-Validation on the training set and holdout evaluation on the test set.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

candidate_models = {
    'XGBoost': XGBRegressor(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42),
    'Lasso Regression': Lasso(alpha=0.1),
    'Ridge Regression': Ridge(alpha=1.0),
    'KNN': KNeighborsRegressor(n_neighbors=7),
    'SVR': SVR(C=10.0, epsilon=0.2)
}

benchmark_records = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in candidate_models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('regressor', model)])
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='r2')
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    
    benchmark_records.append({
        'Model': name,
        'CV R2 (5-Fold)': round(float(np.mean(cv_scores)), 4),
        'Test R2': round(float(r2), 4),
        'RMSE ($)': round(float(rmse), 2),
        'MAE ($)': round(float(mae), 2),
        'MAPE (%)': round(float(mape), 2)
    })

results_df = pd.DataFrame(benchmark_records).sort_values(by='Test R2', ascending=False)
results_df

## 5. Visualizations & Model Evaluation

In [ ]:
# 1. Benchmark Barplot
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=results_df, x='Test R2', y='Model', palette='viridis', ax=ax)
ax.set_title('Holdout R² Comparison Across 7 ML Algorithms', fontsize=13, fontweight='bold')
ax.set_xlim(0.7, 1.02)
for i, val in enumerate(results_df['Test R2']):
    ax.text(val + 0.005, i, f"{val:.4f}", va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Serialization for Production Deployment

In [ ]:
best_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42))
])
best_pipe.fit(X_train, y_train)

# Save artifact
os.makedirs('saved_models', exist_ok=True)
joblib.dump(best_pipe, 'saved_models/best_model.joblib')
print("🎉 Best pipeline (XGBoost) successfully trained and exported for Flask web application.")